In [ ]:
!pip install -q scikit-learn nltk

In [ ]:
import nltk, string, os
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving abs1.txt to abs1.txt
Saving abs3.txt to abs3.txt
Saving abs2.txt to abs2.txt


In [ ]:

from pathlib import Path
docs_raw, names = [], []
for p in sorted(Path('.').glob('abs*.txt')):
    docs_raw.append(p.read_text(encoding='utf-8'))
    names.append(p.stem)
assert len(docs_raw) == 3, "Harus ada 3 file abs1.txt‑abs3.txt"
print("Berhasil memuat:", names)


Berhasil memuat: ['abs1', 'abs2', 'abs3']


In [ ]:

import re, nltk, string, pandas as pd
from nltk.corpus import stopwords

nltk.download("stopwords", quiet=True)
STOP_WORDS = stopwords.words('english') + stopwords.words('indonesian')
PUNCT_RE = re.compile(r"[^\w\s]", flags=re.UNICODE)

def normalize(text: str) -> str:
    text = text.lower()
    text = PUNCT_RE.sub(" ", text)
    return " ".join(text.split())


In [ ]:

from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(stop_words=STOP_WORDS, preprocessor=normalize)
tf_matrix = cv.fit_transform(docs_raw)


tf_df = pd.DataFrame(
    tf_matrix.toarray().T,
    index=cv.get_feature_names_out(),
    columns=names
)


def top_n(df, col, n=10):
    return df[col].sort_values(ascending=False).head(n)

for col in names:
    display(top_n(tf_df, col).to_frame(name=f"freq_{col}"))


/usr/local/lib/python3.11/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['baiknya', 'berkali', 'kali', 'kurangnya', 'mata', 'olah', 'sekurang', 'setidak', 'tama', 'tidaknya'] not in stop_words.
  warnings.warn(


,freq_abs1
ai,10
business,8
adoption,7
performance,6
relationship,3
impact,3
implementation,3
factors,3
artificial,2
challenges,2


,freq_abs2
fencf,6
book,5
cold,4
item,4
start,4
commerce,2
data,2
rating,2
sales,2
also,2


,freq_abs3
wayang,6
culture,5
local,5
kamasan,5
cultural,4
balinese,3
approach,3
machine,3
learning,3
preservation,3


In [ ]:

from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

cv = CountVectorizer(stop_words=STOP_WORDS, preprocessor=normalize)
tf_matrix = cv.fit_transform(docs_raw)

freq_df = pd.DataFrame(
    tf_matrix.toarray().T,
    index=cv.get_feature_names_out(),
    columns=names
).sort_index()

display(freq_df.head(55))


/usr/local/lib/python3.11/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['baiknya', 'berkali', 'kali', 'kurangnya', 'mata', 'olah', 'sekurang', 'setidak', 'tama', 'tidaknya'] not in stop_words.
  warnings.warn(


,abs1,abs2,abs3
04,0,1,0
73,0,1,0
able,0,1,0
acceptance,1,0,0
according,0,1,0
accuracy,0,2,0
accurately,0,1,1
across,0,1,0
actual,0,1,0
addition,0,1,0


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import numpy as np
Second

tfidf_vectorizer = TfidfVectorizer(stop_words=STOP_WORDS, preprocessor=normalize)
tfidf_mat = tfidf_vectorizer.fit_transform(docs_raw)

terms = np.array(tfidf_vectorizer.get_feature_names_out())

def top_terms(doc_idx, k=10):
    vec = tfidf_mat[doc_idx].toarray().ravel()
    top_ids = vec.argsort()[-k:][::-1]
    return pd.DataFrame({
        "term": terms[top_ids],
        "tfidf_score": vec[top_ids]
    })

print("abs1  – Top 5 term (TF‑IDF)")
display(top_terms(names.index("abs1")))

print("abs2  – Top 5 term (TF‑IDF)")
display(top_terms(names.index("abs2")))
print("abs3  – Top 5 term (TF‑IDF)")
display(top_terms(names.index("abs3")))

▶️  abs1  – Top 5 term (TF‑IDF)


/usr/local/lib/python3.11/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['baiknya', 'berkali', 'kali', 'kurangnya', 'mata', 'olah', 'sekurang', 'setidak', 'tama', 'tidaknya'] not in stop_words.
  warnings.warn(


,term,tfidf_score
0,ai,0.506949
1,business,0.405559
2,adoption,0.354864
3,performance,0.304170
4,relationship,0.152085
5,factors,0.152085
6,implementation,0.152085
7,impact,0.152085
8,attention,0.101390
9,ethical,0.101390


▶️  abs2  – Top 5 term (TF‑IDF)


,term,tfidf_score
0,fencf,0.400937
1,book,0.334114
2,start,0.267291
3,item,0.267291
4,cold,0.267291
5,sales,0.133646
6,accuracy,0.133646
7,rating,0.133646
8,information,0.133646
9,commerce,0.133646


▶️  abs3  – Top 5 term (TF‑IDF)


,term,tfidf_score
0,wayang,0.380519
1,local,0.317099
2,kamasan,0.317099
3,cultural,0.253679
4,culture,0.241162
5,machine,0.190259
6,preservation,0.190259
7,learning,0.190259
8,approach,0.190259
9,balinese,0.190259


In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(stop_words=STOP_WORDS, preprocessor=normalize)
tfidf_vectorizer.fit(docs_raw)

def common_terms(doc_idx_a, doc_idx_b, top_k=10):
    """Mengembalikan DataFrame top_k istilah yang muncul di kedua dokumen."""
    terms = np.array(tfidf_vectorizer.get_feature_names_out())

    vec_a = tfidf_mat[doc_idx_a].toarray().ravel()
    vec_b = tfidf_mat[doc_idx_b].toarray().ravel()

    mask = (vec_a > 0) & (vec_b > 0)
    shared_terms = terms[mask]

    score = (vec_a + vec_b)[mask]

    top = np.argsort(score)[-top_k:][::-1]
    df_shared = pd.DataFrame({
        "term": shared_terms[top],
        f"tfidf_{names[doc_idx_a]}": vec_a[mask][top],
        f"tfidf_{names[doc_idx_b]}": vec_b[mask][top],
    })
    return df_shared

for i in range(len(names)):
    for j in range(i+1, len(names)):
        print(f"\n=== {names[i]} ↔ {names[j]} ===")
        display(common_terms(i, j, top_k=5))


=== abs1 ↔ abs2 ===


/usr/local/lib/python3.11/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['baiknya', 'berkali', 'kali', 'kurangnya', 'mata', 'olah', 'sekurang', 'setidak', 'tama', 'tidaknya'] not in stop_words.
  warnings.warn(


,term,tfidf_abs1,tfidf_abs2
0,data,0.077110,0.101641
1,potential,0.077110,0.050820
2,challenges,0.077110,0.050820
3,also,0.029941,0.078933
4,due,0.038555,0.050820



=== abs1 ↔ abs3 ===


,term,tfidf_abs1,tfidf_abs3
0,culture,0.038555,0.241162
1,study,0.038555,0.048232
2,understanding,0.038555,0.048232
3,aspects,0.038555,0.048232
4,findings,0.038555,0.048232



=== abs2 ↔ abs3 ===


,term,tfidf_abs2,tfidf_abs3
0,also,0.078933,0.037457
1,method,0.050820,0.048232
2,accurately,0.050820,0.048232
3,research,0.039467,0.037457


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(stop_words=STOP_WORDS, preprocessor=normalize)
tfidf_mat  = vectorizer.fit_transform(docs_raw)
sim_mat    = cosine_similarity(tfidf_mat)

print(sim_mat.round(4))

[[1.     0.0212 0.019 ]
 [0.0212 1.     0.0093]
 [0.019  0.0093 1.    ]]


/usr/local/lib/python3.11/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['baiknya', 'berkali', 'kali', 'kurangnya', 'mata', 'olah', 'sekurang', 'setidak', 'tama', 'tidaknya'] not in stop_words.
  warnings.warn(


In [ ]:

for i in range(3):
    for j in range(i+1, 3):
        val = sim[i, j]
        label = ("Hampir identik" if val >= 0.9 else
                 "Sangat mirip"    if val >= 0.7 else
                 "Cukup mirip"     if val >= 0.4 else
                 "Sedikit mirip"   if val >= 0.1 else
                 "Nyaris tak berhubungan")
        print(f"{names[i]} ↔ {names[j]} : {val:.2%}  ➜  {label}")


abs1 ↔ abs2 : 2.10%  ➜  Nyaris tak berhubungan
abs1 ↔ abs3 : 1.95%  ➜  Nyaris tak berhubungan
abs2 ↔ abs3 : 0.95%  ➜  Nyaris tak berhubungan


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

cv = CountVectorizer(stop_words=STOP_WORDS, preprocessor=normalize)
tf_matrix = cv.fit_transform(docs_raw)           # hitung frekuensi

tf_df = pd.DataFrame(
    tf_matrix.toarray().T,                       # baris = term, kolom = dokumen
    index=cv.get_feature_names_out(),
    columns=names
)
